In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

os.makedirs('2.3_plots', exist_ok=True)

In [ ]:
#
# Discretisation:
#   du/dt = Du * lap(u) - u*v^2 + f*(1-u)
#   dv/dt = Dv * lap(v) + u*v^2 - (f+k)*v
#
#   Laplacian — 5-point stencil:
#   lap(u)[i,j] = (u[i+1,j]+u[i-1,j]+u[i,j+1]+u[i,j-1] - 4*u[i,j]) / dx^2
#
#   Time integration: explicit forward Euler
#
#   Boundary conditions:
#   To represent U and V reacting and diffusing in a square vessel: U is continuously supplied from the top boundary; The other walls are insulating. 
#   - Neumann at left/right/bottom: 
#       i=0  -> im1=1,   i=N-1 -> ip1=N-2   (same logic for j)
#   - Dirichlet at top row (i=N-1): u=1, v=0, overwritten after every step

@njit
def step(U, V, Du, Dv, f, k, dt, dx):
    N   = U.shape[0]
    U_new = np.empty_like(U)
    V_new = np.empty_like(V)
    dx2   = dx * dx

    for i in range(N - 1):          # top row set by Dirichlet below
        for j in range(N):
            im1 = i - 1 if i > 0     else 1
            ip1 = i + 1 if i < N - 1 else N - 2
            jm1 = j - 1 if j > 0     else 1
            jp1 = j + 1 if j < N - 1 else N - 2

            lap_u = (U[ip1,j] + U[im1,j] + U[i,jp1] + U[i,jm1] - 4*U[i,j]) / dx2
            lap_v = (V[ip1,j] + V[im1,j] + V[i,jp1] + V[i,jm1] - 4*V[i,j]) / dx2

            uvv = U[i,j] * V[i,j] * V[i,j]
            U_new[i,j] = U[i,j] + dt * (Du * lap_u - uvv + f * (1 - U[i,j]))
            V_new[i,j] = V[i,j] + dt * (Dv * lap_v + uvv - (f + k) * V[i,j])

    U_new[N-1, :] = 1.0
    V_new[N-1, :] = 0.0
    return U_new, V_new


def make_initial(N, noise=0.0, seed=42):
    """
    u = 0.5 everywhere.
    v = 0.25 in the central square of 20x20 physical units (= 20/dx grid cells).
    When noise > 0, it is generated on a fixed 100x100 reference grid and
    resampled to size N — so every resolution sees the same physical perturbation.
    """
    rng = np.random.default_rng(seed)

    U = np.full((N, N), 0.5)
    V = np.zeros((N, N))

    # Central square: always 20 physical units wide regardless of dx
    mid = N // 2
    sq  = N // 10          
    V[mid-sq : mid+sq, mid-sq : mid+sq] = 0.25

    if noise > 0:
        ref = 100
        nu  = rng.uniform(-noise, noise, (ref, ref))
        nv  = rng.uniform(-noise, noise, (ref, ref))
        if N != ref:
            idx = (np.linspace(0, ref - 1, N) + 0.5).astype(int).clip(0, ref - 1)
            nu  = nu[np.ix_(idx, idx)]
            nv  = nv[np.ix_(idx, idx)]
        U += nu
        V += nv
        np.clip(U, 0, 1, out=U)
        np.clip(V, 0, 1, out=V)


    U[-1, :] = 1.0
    V[-1, :] = 0.0
    return U, V


def run(N, dt, dx, steps, Du=0.16, Dv=0.08, f=0.035, k=0.060, noise=0.0, seed=42):
    """Update the system. Physical domain = N*dx = 100. Wall clock time = steps*dt."""
    U, V = make_initial(N, noise=noise, seed=seed)
    for _ in range(steps):
        U, V = step(U, V, Du, Dv, f, k, dt, dx)
    return U, V


# Warm up numba JIT
_u, _v = make_initial(4)
step(_u, _v, 0.16, 0.08, 0.035, 0.060, 1.0, 1.0)
print("Ready.")

In [ ]:

# FIGURE 1: Evolution and Symmetry Breaking (U and V)
N = 100
dx = 1.0
dt = 1.0
Du, Dv = 0.16, 0.08
f, k = 0.035, 0.060  
noise_level = 0.05

print("Generating Figure 1 data...")

U_init_noisy, V_init_noisy = make_initial(N, noise=noise_level)


U_1k_noisy, V_1k_noisy = run(N, dt, dx, 1000, Du, Dv, f, k, noise=noise_level)


U_8k_noisy, V_8k_noisy = run(N, dt, dx, 8000, Du, Dv, f, k, noise=noise_level)


U_8k_perfect, V_8k_perfect = run(N, dt, dx, 8000, Du, Dv, f, k, noise=0.0)

# ── Plotting ──────────────────────────────────────────────────────────────────
# Create a 2x4 grid: Top row for U, Bottom row for V (stacked horizontally)
fig, axs = plt.subplots(2, 4, figsize=(16, 8))


# Data mapping for easy loop plotting
plot_data = [
    # Row 0: U Concentration
    [
        (U_init_noisy, "U: t=0 (Initial + Noise)"),
        (U_1k_noisy, "U: t=1000"),
        (U_8k_noisy, "U: t=8000 (With Noise)"),
        (U_8k_perfect, "U: t=8000 (NO Noise)")
    ],
    # Row 1: V Concentration
    [
        (V_init_noisy, "V: t=0 (Initial + Noise)"),
        (V_1k_noisy, "V: t=1000"),
        (V_8k_noisy, "V: t=8000 (With Noise)"),
        (V_8k_perfect, "V: t=8000 (NO Noise)")
    ]
]

# Color settings
cmaps = ['RdPu', 'YlGn']
vmins = [0.0, 0.0]
vmaxs = [1.0, 0.4]  # V concentration typically peaks lower than U

images = []

for row in range(2):
    for col in range(4):
        data, title = plot_data[row][col]
        ax = axs[row, col]
        
        im = ax.imshow(data.T, origin='lower', cmap=cmaps[row], vmin=vmins[row], vmax=vmaxs[row])
        ax.set_title(title, fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])
        
        if col == 3: # Save the image object of the last column for the colorbar
            images.append(im)

# Formatting colorbars for each row
fig.subplots_adjust(right=0.92, hspace=0.2, wspace=0.1)

# Colorbar for U (Top row)
cbar_ax_U = fig.add_axes([0.94, 0.53, 0.015, 0.35])
fig.colorbar(images[0], cax=cbar_ax_U, label='u')

# Colorbar for V (Bottom row)
cbar_ax_V = fig.add_axes([0.94, 0.11, 0.015, 0.35])
fig.colorbar(images[1], cax=cbar_ax_V, label='v')

plt.savefig('2.3_plots/figure1_symmetry_breaking_UV.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# FIGURE 2: Morphological Evolution (Square vs. Cross Seed)
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

# Numba-optimized step function with periodic boundaries, larger grid and more run time for noteworthy patterns. 
@njit
def step_periodic(U, V, Du, Dv, f, k, dt, dx):
    N = U.shape[0]
    U_new, V_new = np.empty_like(U), np.empty_like(V)
    dx2 = dx * dx

    for i in range(N):
        for j in range(N):
            im1, ip1 = (i - 1) % N, (i + 1) % N
            jm1, jp1 = (j - 1) % N, (j + 1) % N

            lap_u = (U[ip1,j] + U[im1,j] + U[i,jp1] + U[i,jm1] - 4*U[i,j]) / dx2
            lap_v = (V[ip1,j] + V[im1,j] + V[i,jp1] + V[i,jm1] - 4*V[i,j]) / dx2

            uvv = U[i,j] * V[i,j]**2
            
            U_new[i,j] = U[i,j] + dt * (Du * lap_u - uvv + f * (1 - U[i,j]))
            V_new[i,j] = V[i,j] + dt * (Dv * lap_v + uvv - (f + k) * V[i,j])

    return U_new, V_new

# Initialize grid with a specific central seed and low background noise
def init_custom_seed(N, seed_type, noise_level=0.005, seed_val=42):
    rng = np.random.default_rng(seed_val)
    U, V = np.ones((N, N)), np.zeros((N, N))
    mid = N // 2
    
    if seed_type == 'square':
        sq = 10 
        U[mid-sq:mid+sq, mid-sq:mid+sq] = 0.5   
        V[mid-sq:mid+sq, mid-sq:mid+sq] = 0.25  
        
    elif seed_type == 'cross':
        # Intersecting 20x6 and 6x20 rectangular bars
        U[mid-10:mid+10, mid-3:mid+3] = 0.5
        V[mid-10:mid+10, mid-3:mid+3] = 0.25
        U[mid-3:mid+3, mid-10:mid+10] = 0.5
        V[mid-3:mid+3, mid-10:mid+10] = 0.25

    U += rng.uniform(-noise_level, noise_level, (N, N))
    V += rng.uniform(-noise_level, noise_level, (N, N))
    np.clip(U, 0, 1, out=U)
    np.clip(V, 0, 1, out=V)
    
    return U, V

# Setup parameters
N, dt, dx = 256, 1.0, 1.0
steps = 15000 
noise_val = 0.005

configs = [
    {"Du": 0.16, "Dv": 0.08, "F": 0.060, "k": 0.0609},
    {"Du": 0.16, "Dv": 0.08, "F": 0.060, "k": 0.0613},
    {"Du": 0.20, "Dv": 0.10, "F": 0.030, "k": 0.0620},
    {"Du": 0.10, "Dv": 0.05, "F": 0.026, "k": 0.0510}
]

print("Simulating Square vs Cross seeds (256x256 grid)...")
results_square, results_cross = [], []

for cfg in configs:
    print(f"  Running F={cfg['F']:.3f}, k={cfg['k']:.4f}...")
    
    U_sq, V_sq = init_custom_seed(N, 'square', noise_val)
    U_cr, V_cr = init_custom_seed(N, 'cross', noise_val)
    
    # Run both shapes simultaneously for the current config
    for _ in range(steps):
        U_sq, V_sq = step_periodic(U_sq, V_sq, cfg['Du'], cfg['Dv'], cfg['F'], cfg['k'], dt, dx)
        U_cr, V_cr = step_periodic(U_cr, V_cr, cfg['Du'], cfg['Dv'], cfg['F'], cfg['k'], dt, dx)
        
    results_square.append(U_sq)
    results_cross.append(U_cr)

# Plotting
fig, axs = plt.subplots(2, 4, figsize=(18, 9))


axs[0, 0].set_ylabel("Square Seed", fontsize=14, labelpad=10)
axs[1, 0].set_ylabel("Cross (+) Seed", fontsize=14, labelpad=10)

images = []
for col, cfg in enumerate(configs):
    param_label = f"Du={cfg['Du']}, Dv={cfg['Dv']}\nF={cfg['F']:.3f}, k={cfg['k']:.4f}"
    
    # Top Row: Square Seed
    ax_top = axs[0, col]
    im_top = ax_top.imshow(results_square[col].T, origin='lower', cmap='RdPu', vmin=0.0, vmax=1.0)
    ax_top.set_title(param_label, fontsize=11, pad=10)
    ax_top.set_xticks([]); ax_top.set_yticks([])
    
    # Bottom Row: Cross Seed
    ax_bot = axs[1, col]
    im_bot = ax_bot.imshow(results_cross[col].T, origin='lower', cmap='RdPu', vmin=0.0, vmax=1.0)
    ax_bot.set_xticks([]); ax_bot.set_yticks([])
    
    if col == 3:
        images.append(im_top)

fig.subplots_adjust(right=0.92, hspace=0.15, wspace=0.05)
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(images[0], cax=cbar_ax, label='U Concentration')

os.makedirs('2.3_plots', exist_ok=True)
plt.savefig('2.3_plots/figure2_dramatic_seed_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# FIGURE 3: Testing Grid Resolution 
# ==============================================================================

f, k = 0.035, 0.060 
Du, Dv = 0.16, 0.08
noise_level = 0.05  
physical_time = 8000

res_configs = [
    dict(dx=2.0, dt=4.00, label="Coarse grid (dx=2.0)"),
    dict(dx=1.0, dt=1.00, label="Normal grid (dx=1.0)"),
    dict(dx=0.5, dt=0.25, label="Fine grid (dx=0.5)"),
]

print("Simulating data for Figure 3...")
res_results = []

for cfg in res_configs:
    dx_ = cfg['dx']
    dt_ = cfg['dt']
    

    N_ = int(round(100.0 / dx_))
    steps_ = int(round(physical_time / dt_))
    

    U_, V_ = run(N_, dt_, dx_, steps_, Du, Dv, f, k, noise=noise_level)
    res_results.append((cfg['label'], U_, N_, dx_))

# -- Setup the Plot --
fig, axes = plt.subplots(1, 3, figsize=(12, 4))


for col, (label, U_, N_, dx_) in enumerate(res_results):
    physical_size = N_ * dx_
    

    im = axes[col].imshow(
        U_.T, 
        origin='lower', 
        cmap='RdPu', 
        vmin=0, 
        vmax=1, 
        extent=[0, physical_size, 0, physical_size]
    )
    axes[col].set_title(label, fontsize=11)
    axes[col].set_xticks([])
    axes[col].set_yticks([])

# Add a single colorbar
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label='u')

plt.savefig('2.3_plots/figure3_resolution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
#animation

N, dt, dx = 100, 1.0, 1.0
f_anim, k_anim  = 0.020, 0.050
frames          = 80
steps_per_frame = 150

print("Initialising ...")
U_a, V_a = make_initial(N, noise=0.02, seed=0)

print("Burn-in 3000 steps ...")
for _ in range(3000):
    U_a, V_a = step(U_a, V_a, 0.16, 0.08, f_anim, k_anim, dt, dx)

fig_a, ax_a = plt.subplots(figsize=(4, 4))
ax_a.axis('off')
im_a  = ax_a.imshow(V_a.T, origin='lower', cmap='YlGn', vmin=0, vmax=0.4, animated=True)
ttl_a = ax_a.set_title('', fontsize=8)

def update(frame):
    global U_a, V_a
    for _ in range(steps_per_frame):
        U_a, V_a = step(U_a, V_a, 0.16, 0.08, f_anim, k_anim, dt, dx)
    im_a.set_data(V_a.T)
    ttl_a.set_text(f"V  f={f_anim}, k={k_anim}   t={3000 + (frame+1)*steps_per_frame}")
    return im_a, ttl_a

ani = FuncAnimation(fig_a, update, frames=frames, interval=80, blit=True)
ani.save('2.3_plots/plot4_animation.gif', writer=PillowWriter(fps=12))
plt.close(fig_a)
print("Saved plots/plot4_animation.gif")

Image('2.3_plots/plot4_animation.gif')
